# **Multimodal Customer Issue and Receipt Investigator**

> **Description:** This execution script imports the modularized classes from `utilis` It loads the customer support ticket dataset, initializes the in-memory RAG vector index, processes incoming receipt/screenshot images via OCR in batches, and runs the full multimodal pipeline to generate AI-driven responses and insights

In [1]:
import sys

from pathlib import Path
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
folder_path = str(BASE_DIR / "src")
if folder_path not in sys.path:
    sys.path.append(folder_path)

from utilis import DataCollection, ImageOCRProcessing, InMemoryRAGEngine, LLMGenerator, FinalMultimodalRAGPipeline

c:\Users\Hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
rag_system = FinalMultimodalRAGPipeline(
    csv_path = str(BASE_DIR / "data" / "customer_support_tickets.csv"), 
    text_column = "Ticket Description"
)

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


[INFO] Loading EasyOCR reader...


c:\Users\Hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


[INFO] Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3115.11it/s]


[INFO] Loading Hugging Face LLM: gpt2...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2418.24it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[INFO] Loaded 8469 documents into knowledge base
[INFO] Generating embeddings for knowledge base...


Batches: 100%|██████████| 265/265 [02:48<00:00,  1.58it/s]

[INFO] Index built successfully with 8469 items


In [3]:
from pathlib import Path

images_dir = BASE_DIR / "data" / "sample_images"

image_extensions = {".jpg", ".jpeg", ".png"}
image_paths = [file for file in images_dir.iterdir() if file.suffix.lower() in image_extensions]

print(f"[INFO] Total {len(image_paths)} images sre for processing .\n")

batch_results = []
for i, img_path in enumerate(image_paths, start=1):
    print(f"[{i}/{len(image_paths)}] Processing image: {img_path.name}")
    
    response = rag_system.run(
        user_question="What issue is reported in the customer support ticket?",
        image_path=str(img_path)
    )
    
    batch_results.append({
        "image_name": img_path.name,
        "ai_response": response
    })

print("\n--- Batch Processing Completed! ---")

[INFO] Total 6 images sre for processing .

[1/6] Processing image: img-1.png


c:\Users\Hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[INFO] OCR Extracted Text: 'TECHGADGETS INC. March 22, 2021 Marisa Obrien 123 Maple St Mumbai; India. Ix GoPro Hero S399.99 Ix LG Smart TV S1250.00 Ix Dell XPS Laptop S850.00 Subtotal: S2499.99 Total: S2450.61 (Including discount) PAID'

[INFO] Searching database for: 'What issue is reported in the customer support ticket? Image Details: TECHGADGETS INC. March 22, 2021 Marisa Obrien 123 Maple St Mumbai; India. Ix GoPro Hero S399.99 Ix LG Smart TV S1250.00 Ix Dell XPS Laptop S850.00 Subtotal: S2499.99 Total: S2450.61 (Including discount) PAID'
[INFO] Generating final answer with LLM...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[2/6] Processing image: img-2.png


...

[INFO] OCR Extracted Text: 'TECHGADGETS INC. May 22,2021 To: Jessica Rios 789 Oak Ave Kolkala, India Ajogunl Aintunt loce Ihon EOS Camera . 81909.99 Ix Canon 1X Mernory Card - S50,00 ix Camera 8ag ~ Sublotal 32080.09 Incuthg hux PENDING] RREFUNDL 5120,00 S2450.01 Total'

[INFO] Searching database for: 'What issue is reported in the customer support ticket? Image Details: TECHGADGETS INC. May 22,2021 To: Jessica Rios 789 Oak Ave Kolkala, India Ajogunl Aintunt loce Ihon EOS Camera . 81909.99 Ix Canon 1X Mernory Card - S50,00 ix Camera 8ag ~ Sublotal 32080.09 Incuthg hux PENDING] RREFUNDL 5120,00 S2450.01 Total'
[INFO] Generating final answer with LLM...
[5/6] Processing image: img-5.png


c:\Users\Hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[INFO] OCR Extracted Text: 'Support Chat - Marisa Obrien Support Agent Hello Marisa, understand you're having trouble with Microsoft Office setup: Marisa Obrien Yes; the installation failed. It says "Error 1603: Fatal error during installation" . LeatLni 1224497 6l  Tna'

[INFO] Searching database for: 'What issue is reported in the customer support ticket? Image Details: Support Chat - Marisa Obrien Support Agent Hello Marisa, understand you're having trouble with Microsoft Office setup: Marisa Obrien Yes; the installation failed. It says "Error 1603: Fatal error during installation" . LeatLni 1224497 6l  Tna'
[INFO] Generating final answer with LLM...
[6/6] Processing image: img-6.png


c:\Users\Hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[INFO] OCR Extracted Text: 'CRM Goneko Fren TICKET Id; CRITICAL CAHCELLATIOK ReIfiEST Qudoner Ortlaha Sttrs Molialg Potelen Enet CcULrugccoapk(On Lu Se Fiod | Pidasct Enskcan Drat cobklar 68 Yad Edti Kgo] #itts Cratoei Customer calied Qdtet "Fcund bentgr Oca . Oflered ' Laltnet Resson. Status exct Iv Cukael discount but ( Mand 68460 - Icncanceici nuidani C liderten declined = EGis'

[INFO] Searching database for: 'What issue is reported in the customer support ticket? Image Details: CRM Goneko Fren TICKET Id; CRITICAL CAHCELLATIOK ReIfiEST Qudoner Ortlaha Sttrs Molialg Potelen Enet CcULrugccoapk(On Lu Se Fiod | Pidasct Enskcan Drat cobklar 68 Yad Edti Kgo] #itts Cratoei Customer calied Qdtet "Fcund bentgr Oca . Oflered ' Laltnet Resson. Status exct Iv Cukael discount but ( Mand 68460 - Icncanceici nuidani C liderten declined = EGis'
[INFO] Generating final answer with LLM...

--- Batch Processing Completed! ---
